In [8]:
print ("Angga Anggieanie")
print ("NIM : 250401020172")
print ("Pertemuan ke 3")

# LANGKAH 0: Import library
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests
from pandas import json_normalize
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimpor!')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')
# Muat dataset dari Google Drive (link dari Modul Pertemuan 3)
URL_GDRIVE = 'https://drive.google.com/file/d/1LfQWProB0VjWN5q8bKuRIgn-stULfIRo/view'

try:
    df = pd.read_csv(URL_GDRIVE)
    print(f'Dataset berhasil dimuat dari Google Drive!')
except Exception:
    # Fallback: buat dataset simulasi jika Google Drive tidak dapat diakses
    print('Gagal memuat dari GDrive, membuat dataset simulasi...')
    np.random.seed(42)
    n = 130
    kota_list  = ['jakarta', 'Jakarta', 'JAKARTA', 'bandung', 'Bandung', 'surabaya', 'Surabaya']
    kondisi_list = ['Baik', 'baik', 'BAIK', 'sedang', 'Sedang', 'buruk', 'Buruk']
    data = {
        'id'           : range(1, n + 1),
        'luas_m2'      : np.where(np.random.rand(n) < 0.10, np.nan,
                                  np.random.uniform(36, 250, n).round(1)),
        'harga_juta'   : np.where(np.random.rand(n) < 0.08, np.nan,
                                  np.random.uniform(200, 5000, n).round(1)),
        'kota'         : np.random.choice(kota_list, n),
        'kamar'        : np.where(np.random.rand(n) < 0.07, np.nan,
                                  np.random.choice([2, 3, 4, 5], n)),
        'tahun_bangun' : np.random.choice(
                            list(range(1990, 2024)) + [1900, 1800, 9999], n),
        'kondisi'      : np.random.choice(kondisi_list, n),
    }
    df = pd.DataFrame(data)
    # Tambahkan outlier harga
    df.loc[5, 'harga_juta']  = 999999
    df.loc[12, 'harga_juta'] = -500
    # Tambahkan duplikat
    df = pd.concat([df, df.iloc[[0, 1, 2]]], ignore_index=True)
    print('Dataset simulasi berhasil dibuat!')

print(f'\nShape awal  : {df.shape}')
print(f'Kolom       : {df.columns.tolist()}')
# LANGKAH 1a: Info dasar
print('=== INFORMASI DATASET ===')
print(f'Jumlah baris  : {df.shape[0]}')
print(f'Jumlah kolom  : {df.shape[1]}')
print()
df.info()
# LANGKAH 1b: Lima baris pertama
print('=== LIMA BARIS PERTAMA ===')
df.head(10)
# LANGKAH 1c: Statistik deskriptif
print('=== STATISTIK DESKRIPTIF ===')
df.describe().round(2)
# LANGKAH 1d: Audit missing values
print('=== AUDIT MISSING VALUES ===')
missing_count = df.isnull().sum()
missing_pct   = (df.isnull().sum() / len(df) * 100).round(2)

audit_missing = pd.DataFrame({
    'Jumlah Missing' : missing_count,
    'Persentase (%)'  : missing_pct
})
print(audit_missing)
print(f'\nTotal sel yang missing: {df.isnull().sum().sum()}')
# LANGKAH 1e: Cek duplikat dan nilai unik kolom kategorik
print(f'Jumlah baris duplikat: {df.duplicated().sum()}')
print()
print('Nilai unik kolom KOTA:')
print(df['kota'].value_counts())
print()
print('Nilai unik kolom KONDISI:')
print(df['kondisi'].value_counts())
# LANGKAH 2: Hapus data duplikat
print(f'Shape sebelum hapus duplikat: {df.shape}')
print(f'Jumlah baris duplikat       : {df.duplicated().sum()}')

# Tampilkan baris yang duplikat
df_dup = df[df.duplicated(keep=False)]
if len(df_dup) > 0:
    print(f'\nBaris yang duplikat (semua kemunculan):')
    print(df_dup)

# Hapus duplikat
df.drop_duplicates(keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nShape setelah hapus duplikat: {df.shape}')
print(f'Sisa baris duplikat         : {df.duplicated().sum()}')
# LANGKAH 3: Normalisasi string
print('=== SEBELUM NORMALISASI ===')
print('Nilai unik KOTA   :', df['kota'].unique())
print('Nilai unik KONDISI:', df['kondisi'].unique())

# Normalisasi kolom kota → Title Case
df['kota'] = df['kota'].str.strip().str.title()

# Normalisasi kolom kondisi → lower case
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('\n=== SETELAH NORMALISASI ===')
print('Nilai unik KOTA   :', df['kota'].unique())
print('Nilai unik KONDISI:', df['kondisi'].unique())

print('\nDistribusi KOTA:')
print(df['kota'].value_counts())

# LANGKAH 4: Imputasi missing values
print('=== MISSING SEBELUM IMPUTASI ===')
print(df[['luas_m2', 'harga_juta', 'kamar']].isnull().sum())

# Imputasi median untuk kolom numerik
median_luas   = df['luas_m2'].median()
median_harga  = df['harga_juta'].median()
modus_kamar   = df['kamar'].mode()[0]

print(f'\nNilai imputasi:')
print(f'  Median luas_m2    : {median_luas:.1f} m²')
print(f'  Median harga_juta : Rp {median_harga:.1f} juta')
print(f'  Modus kamar       : {modus_kamar} kamar')

# Terapkan imputasi
df['luas_m2']    = df['luas_m2'].fillna(median_luas)
df['harga_juta'] = df['harga_juta'].fillna(median_harga)
df['kamar']      = df['kamar'].fillna(modus_kamar)

print('\n=== MISSING SETELAH IMPUTASI ===')
print(df[['luas_m2', 'harga_juta', 'kamar']].isnull().sum())
print(f'\nTotal missing yang tersisa: {df.isnull().sum().sum()}')
# LANGKAH 5: Deteksi dan tangani outlier menggunakan IQR Fence + capping

def iqr_fence_info(series, nama_kolom):
    """Menampilkan informasi IQR Fence dan jumlah outlier."""
    Q1  = series.quantile(0.25)
    Q3  = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f'\n--- {nama_kolom} ---')
    print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
    print(f'  Batas bawah : {lower:.2f}')
    print(f'  Batas atas  : {upper:.2f}')
    print(f'  Outlier     : {len(outliers)} baris')
    return lower, upper

print('=== DETEKSI OUTLIER (SEBELUM CAPPING) ===')
target_cols = ['harga_juta', 'luas_m2', 'tahun_bangun']
batas = {}
for col in target_cols:
    lower, upper = iqr_fence_info(df[col], col)
    batas[col] = (lower, upper)

print('\n=== TERAPKAN CAPPING ===')
for col in target_cols:
    lo, hi = batas[col]
    before_min, before_max = df[col].min(), df[col].max()
    df[col] = df[col].clip(lower=lo, upper=hi)
    after_min,  after_max  = df[col].min(), df[col].max()
    print(f'{col}: [{before_min:.1f}, {before_max:.1f}] → [{after_min:.1f}, {after_max:.1f}]')

# Cek outlier setelah capping
print('=== VERIFIKASI OUTLIER SETELAH CAPPING ===')
for col in target_cols:
    lo, hi = batas[col]
    sisa = df[(df[col] < lo) | (df[col] > hi)]
    print(f'{col}: sisa outlier = {len(sisa)} baris')
# LANGKAH 6: Validasi akhir
print('=== VALIDASI AKHIR ===')

total_missing  = df.isnull().sum().sum()
total_duplikat = df.duplicated().sum()

assert total_missing  == 0, f'GAGAL: Masih ada {total_missing} missing values!'
assert total_duplikat == 0, f'GAGAL: Masih ada {total_duplikat} duplikat!'

print(f'✓ Total missing values : {total_missing}')
print(f'✓ Total duplikat       : {total_duplikat}')
print(f'✓ Shape akhir dataset  : {df.shape}')
print('\n=== STATISTIK AKHIR ===')
df.describe().round(2)

# Ekspor dataset bersih
NAMA_FILE = 'housing_clean.csv'
df.to_csv(NAMA_FILE, index=False)
print(f'Dataset bersih berhasil disimpan sebagai "{NAMA_FILE}"')
print(f'Ukuran file : {df.shape[0]} baris × {df.shape[1]} kolom')

# LANGKAH 7a: Ambil data dari endpoint /users
BASE_URL = 'https://jsonplaceholder.typicode.com'

print('Mengirim GET request ke /users ...')
response = requests.get(f'{BASE_URL}/users', timeout=10)

print(f'Status Code: {response.status_code}')

if response.status_code == 200:
    data_users = response.json()
    df_users = json_normalize(data_users, sep='_')
    print(f'Berhasil! {len(df_users)} pengguna diterima.')
    print(f'Kolom: {df_users.columns.tolist()}')
else:
    print(f'Error: {response.status_code}')


# Tampilkan subset kolom yang relevan
kolom_tampil = ['id', 'name', 'username', 'email', 'address_city', 'company_name']
print('=== DATA PENGGUNA DARI API ===')
df_users[kolom_tampil]

# LANGKAH 7b: Ambil postingan dari user tertentu (query parameter)
USER_ID = 1
params  = {'userId': USER_ID}

resp_posts = requests.get(f'{BASE_URL}/posts', params=params, timeout=10)
print(f'Status Code: {resp_posts.status_code}')

if resp_posts.status_code == 200:
    df_posts = pd.DataFrame(resp_posts.json())
    print(f'Postingan dari user {USER_ID}: {len(df_posts)} post')
    print(df_posts[['id', 'userId', 'title']].head())
else:
    print(f'Error: {resp_posts.status_code}')
# Eksplorasi dasar DataFrame dari API
print('=== STATISTIK DESKRIPTIF DATA PENGGUNA ===')
print(f'Total pengguna  : {len(df_users)}')
print(f'Total kolom     : {len(df_users.columns)}')
print(f'Missing values  : {df_users.isnull().sum().sum()}')
print()
print('Distribusi berdasarkan kota:')
print(df_users['address_city'].value_counts())

Angga Anggieanie
NIM : 250401020172
Pertemuan ke 3
Library berhasil diimpor!
Pandas  : 2.2.2
NumPy   : 2.0.2
Gagal memuat dari GDrive, membuat dataset simulasi...
Dataset simulasi berhasil dibuat!

Shape awal  : (133, 7)
Kolom       : ['id', 'luas_m2', 'harga_juta', 'kota', 'kamar', 'tahun_bangun', 'kondisi']
=== INFORMASI DATASET ===
Jumlah baris  : 133
Jumlah kolom  : 7

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            133 non-null    int64  
 1   luas_m2       117 non-null    float64
 2   harga_juta    117 non-null    float64
 3   kota          133 non-null    object 
 4   kamar         123 non-null    float64
 5   tahun_bangun  133 non-null    int64  
 6   kondisi       133 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.4+ KB
=== LIMA BARIS PERTAMA ===
=== STATISTIK DESKRIPTIF ===
=== AUDIT MISSI